# Feature Engineering y Preprocesamiento

En este notebook tomaremos los datos consolidados y crearemos las características temporales, de imputación y rezagos (lags) necesarias para entrenar modelos predictivos (como LightGBM).

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

# Carga de datos base
def load_and_merge_data() -> pd.DataFrame:
    train = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))
    stores = pd.read_csv(os.path.join(RAW_DIR, "stores.csv"))
    features = pd.read_csv(os.path.join(RAW_DIR, "features.csv"))
    
    train['Date'] = pd.to_datetime(train['Date'])
    features['Date'] = pd.to_datetime(features['Date'])
    
    merged = train.merge(stores, on='Store', how='left')
    merged = merged.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    
    return merged.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)

df = load_and_merge_data()
print(f"Tamaño del dataset inicial: {df.shape}")

## 1. Tratamiento de Valores Nulos (MarkDowns)
Las promociones tienen muchos valores nulos antes de finales de 2011. Llenaremos los NaN con 0 y crearemos variables binarias (`Has_MarkDown_X`) para indicar si la promoción existía. Esto evita que los modelos asuman erróneamente el cero como un valor normal.

In [ ]:
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

for col in markdown_cols:
    # Crear flag indicando si la promo existía (1) o no (0)
    df[f'Has_{col}'] = np.where(df[col].notnull(), 1, 0)
    # Llenar nulos con 0
    df[col] = df[col].fillna(0)

df[markdown_cols + [f'Has_{c}' for c in markdown_cols]].head()

## 2. Variables Temporales
Extraemos información de la fecha para que nuestro modelo (ej. LightGBM) pueda capturar estacionalidad intrínseca que no viene en los lags (meses de altas compras, fin de año, etc.).

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsHoliday'] = df['IsHoliday'].astype(int)

df[['Date', 'Year', 'Month', 'Week', 'IsHoliday']].head()

## 3. Lags y Features Históricos (Prevención de Leakage)
Calculamos las ventas históricas de ese mismo departamento. Para predecir un horizonte de 39 semanas sin incurrir en data leakage (usar datos del futuro), usamos lags estacionales seguros: 52 semanas (1 año), 53 semanas, y 56 semanas.
**Importante:** Debemos ordenar de forma ascendente por fecha, y agrupar firmemente por `Store` y `Dept`.

In [ ]:
# Garantizar el orden cronológico absoluto por bloque
df = df.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)

# Definimos qué saltos de semanas analizaremos
# Lags: 1 año (52), 1 año + 1 sem (53), 1 mes antes del año pasado (56)
lag_periods = [52, 53, 56]

for lag in lag_periods:
    df[f'Weekly_Sales_lag_{lag}'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(lag)

# Visualizar final del dataset (lag 52 debería estar presente pues hay más de 1 año de datos)
df[['Store', 'Dept', 'Date', 'Weekly_Sales', 'Weekly_Sales_lag_52', 'Weekly_Sales_lag_56']].tail(10)

## 4. Guardar Datos Procesados
Exportamos la base de datos resultante a la carpeta `data/processed/`. Si las variables temporales crearon datos no válidos al principio de la serie (ej. lags con NaNs), los dejaremos por ahora: XGBoost o LightGBM los saben manejar.

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path_csv = os.path.join(PROCESSED_DIR, "train_features.csv")

df.to_csv(output_path_csv, index=False)
print(f"Features generadas correctamente y guardadas en: {output_path_csv}")